In [1]:
import os
import glob
import yaml

import pandas as pd

import tifffile

import zarr
import napari
import dask.array as da

from utils.utility_functions import single_channel_pyramid

In [2]:
# I/O

# read single-cell data
main = pd.read_csv(os.path.join(os.getcwd(), '../input/main.csv'))

# read OME-TIFF, segmentation outlines, and H&E channels
tif_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_image.ome.tif')
seg_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_seg_outlines.ome.tif')
he_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_hema_eosin.ome.tif')

# import markers.csv
markers = pd.read_csv(os.path.join(os.getcwd(), '../input/CyCIF-1A_mcmicro_markers.csv'))

# import image contrast settings
with open(os.path.join(os.getcwd(), '../input/CyCIF-1A_cylinter_contrast_limits.yml')) as f:
    contrast_limits = yaml.safe_load(f)

# the parquet file at the path below is being read because "main.csv" 
# uses trimmed marker channel names as column headers that differ from the raw channel names used 
# in the markers.csv file, which is itself used to index channels in the OME-TIFF image.
for_channels = pd.read_parquet(
    os.path.join(os.getcwd(), '../input/CyCIF-1A_clean_cylinter_clustering_3d_leiden.parquet')
)

# isolate antibodies of interest
abx_channels = [i for i in for_channels.columns if 'nucleiRingMask' in i if 'Hoechst' not in i]

In [3]:
# add H&E image to Napari viewer as separate RGB channels
for color, channel in zip(['red', 'green', 'blue'], [0, 1, 2]):

    img, min, max = single_channel_pyramid(glob.glob(he_path)[0], channel=channel)

    if channel == 0:
        viewer = napari.view_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )
    else:
        viewer.add_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )

In [4]:
# OPTIONAL: add H&E image to Napari viewer as a single channel image

# from lazy_ops import DatasetView
# tiff = tifffile.TiffFile(he_path, is_ome=False)
# pyramid = [
#     zarr.open(tiff.series[0].levels[0].aszarr())[i] for i in
#     list(range(len(tiff.series[0].levels)))
#     ]
# pyramid = [DatasetView(i).lazy_transpose([1, 2, 0]) for i in pyramid]
# pyramid = [da.from_zarr(z) for z in pyramid]
#
# viewer = napari.view_image(pyramid, rgb=True, name='H&E')

In [5]:
# add DNA1 channel to image viewer
dna, min, max = single_channel_pyramid(glob.glob(tif_path)[0], channel=0)
viewer.add_image(
    dna, rgb=False, blending='additive',
    colormap='gray', visible=True, opacity=0.8,
    name='DNA1', contrast_limits=(min, max)
)

<Image layer 'DNA1' at 0x170756430>

In [6]:
# add marker channels to image viewer and apply previously defined contrast limits
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    channel_number = markers['channel_number'][markers['marker_name'] == ch]

    img, min, max = single_channel_pyramid(
        glob.glob(tif_path)[0], channel=(channel_number.item() - 1)
    )
    viewer.add_image(
        img, rgb=False, blending='additive', colormap='lime', visible=False, name=ch,
        contrast_limits=(min, max)
    )
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    viewer.layers[ch].contrast_limits = (
        contrast_limits[ch][0], contrast_limits[ch][1])

In [7]:
total_S13 = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 13)]

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 13) & (main['VAE9_VIG7'] == 4)]
print(f'V4 cells represent {len(centroids)/len(total_S13):.2%} of S13 cells.')
viewer.add_points(
    centroids, name='Seg13_V4', face_color='#00aaff', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 13) & (main['VAE9_VIG7'] == 15)]
print(f'V15 cells represent {len(centroids)/len(total_S13):.2%} of S13 cells.')
viewer.add_points(
    centroids, name='Seg13_V15', face_color='#ffaa00', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

V4 cells represent 39.54% of S13 cells.
V15 cells represent 27.38% of S13 cells.


<Points layer 'Seg13_V15' at 0x176c18640>

In [8]:
# add segmentation outlines to image viewer
seg, min, max = single_channel_pyramid(glob.glob(seg_path)[0], channel=0)
viewer.add_image(
    seg, rgb=False, blending='additive',
    colormap='gray', visible=False,
    name='segmentation', opacity=0.3, contrast_limits=(min, max)
)

<Image layer 'segmentation' at 0x176c09640>

In [9]:
# run image viewer
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'um'